# Reto ViajaFácil · Parte B: impleméntalo
**Versión estudiante · Trabajo individual · 60 minutos**

En la Parte A diseñaste la arquitectura en papel. Ahora implementas una versión pequeña **reutilizando código del curso**:

| Lo que vas a construir | Lo copias de… |
|---|---|
| Agentes con `create_agent` y sus herramientas | Notebook 06, `crear_agente_creditos()` · notebook 06a, `crear_especialista()` |
| Un agente envuelto como herramienta de otro agente | Notebook 06, `consultar_agente_creditos` (paso 9) |
| Un agente principal con varias herramientas | Notebook 06, `crear_asesor()` (paso 10) |
| Aprobación humana antes de reservar | Notebook de Reembolsos (humano en el bucle) |
| Contador de llamadas y Gradio con diagrama | Ya están hechos aquí |

```
Cliente ⇄ Coordinador ─┬─► Agente Reservas  (buscar_vuelos, buscar_hoteles)
                       ├─► Agente Políticas (buscar_politica)   ← en el 06a esto sería RAG
                       ├─► Agente Clima     (consultar_clima)   ← en el 06 esto sería A2A
                       └─► calcular_presupuesto (Python)
            ✋ una persona aprueba ──► reservar()
```

### ✏️ Qué tienes que hacer
Ejecuta las celdas en orden. Hay **7 huecos** `___` marcados con `# ✏️ COMPLETA`. Cada paso dice de dónde copiar y cuál es la salida esperada.

> 🔑 `GROQ_API_KEY` en Secrets de Colab. Todos los datos son ficticios.

## Paso 0 · ▶️ Conectar y preparar el taxímetro (ya está hecho)

In [ ]:
%pip install -q langchain langgraph langchain-groq

In [ ]:
import os, time, json
from langchain_groq import ChatGroq
from langchain_core.callbacks import BaseCallbackHandler
from langchain.tools import tool
from langchain.agents import create_agent

try:
    from google.colab import userdata
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
except ImportError:
    pass

MODELO = 'openai/gpt-oss-120b'

class Contador(BaseCallbackHandler):          # "taxímetro": cuenta cada vez que el LLM responde
    def __init__(self): self.reiniciar()
    def reiniciar(self): self.llamadas, self.tokens = 0, 0
    def on_llm_end(self, response, **kwargs):
        self.llamadas += 1
        try: self.tokens += (response.generations[0][0].message.usage_metadata or {}).get('total_tokens', 0)
        except Exception: pass

CONTADOR = Contador()
def nuevo_llm():
    return ChatGroq(model=MODELO, temperature=0, reasoning_effort='low', callbacks=[CONTADOR])

BITACORA = []
def registrar(de, para, tipo, accion):
    BITACORA.append({'de': de, 'para': para, 'tipo': tipo, 'accion': accion})

print(nuevo_llm().invoke('Responde solo: conectado').content)

## Paso 1 · ▶️ Datos y herramientas de ViajaFácil (ya están hechas)
Léelas: vas a usarlas en los pasos siguientes. Fíjate en el **docstring** de cada una: es lo que lee el LLM para decidir cuándo usarla.

| Herramienta | Qué hace | En la vida real sería… |
|---|---|---|
| `buscar_vuelos`, `buscar_hoteles` | Consultan el sistema de reservas | Un servidor **MCP** |
| `buscar_politica` | Busca en el manual por tema | **RAG** (notebook 06a) |
| `consultar_clima` | Pide el clima | Un agente externo por **A2A** (notebook 06) |
| `calcular_presupuesto` | Suma y compara con el presupuesto | Python (el LLM **no** suma) |
| `reservar` | Hace la reserva | Solo con **aprobación humana** |

In [ ]:
VUELOS = [{'aerolinea': 'Aerolínea Sol', 'precio_por_persona': 420000},
          {'aerolinea': 'Aerolínea Brisa', 'precio_por_persona': 350000}]
HOTELES = [{'hotel': 'Hostal Centro', 'precio_noche': 180000},
           {'hotel': 'Hotel Mar Azul', 'precio_noche': 380000},
           {'hotel': 'Hotel Murallas', 'precio_noche': 650000}]
POLITICAS = {
    'cancelacion': 'Cancelación gratuita hasta 72 horas antes del viaje; después se cobra el 30 %.',
    'equipaje': 'Incluye una maleta de mano de 10 kg por persona. La maleta de bodega cuesta $90.000.',
    'reembolso': 'Los reembolsos se hacen en máximo 15 días hábiles al mismo medio de pago.',
}

@tool
def buscar_vuelos(destino: str) -> list:
    """Busca vuelos de ida y regreso desde Medellín hacia el destino. Precio por persona en COP."""
    registrar('Agente Reservas', 'Agente Reservas', 'Herramienta', 'buscar_vuelos')
    return VUELOS

@tool
def buscar_hoteles(destino: str) -> list:
    """Busca hoteles disponibles en el destino. Precio por noche en COP (habitación doble)."""
    registrar('Agente Reservas', 'Agente Reservas', 'Herramienta', 'buscar_hoteles')
    return HOTELES

@tool
def buscar_politica(tema: str) -> str:
    """Busca en el manual de la agencia. Temas: cancelacion, equipaje, reembolso."""
    registrar('Agente Políticas', 'Agente Políticas', 'Herramienta', f'buscar_politica: {tema}')
    return POLITICAS.get(tema.lower().replace('ó', 'o'), 'El manual no tiene esa información.')

@tool
def consultar_clima(destino: str, mes: str) -> str:
    """Consulta el clima esperado en un destino y mes."""
    registrar('Agente Clima', 'Agente Clima', 'Herramienta', 'consultar_clima')
    return f'{destino} en {mes}: soleado, 31 °C, poca lluvia (dato ficticio).'

@tool
def calcular_presupuesto(precio_vuelo_persona: int, personas: int, precio_hotel_noche: int, noches: int, presupuesto: int) -> dict:
    """Calcula el total del viaje (vuelos + hotel) y dice si cabe en el presupuesto."""
    registrar('Coordinador', 'Coordinador', 'Herramienta', 'calcular_presupuesto')
    total = precio_vuelo_persona * personas + precio_hotel_noche * noches
    return {'total_cop': total, 'dentro_del_presupuesto': total <= presupuesto, 'sobrante_cop': presupuesto - total}

def reservar(vuelo, hotel):
    registrar('Coordinador', 'Coordinador', 'Herramienta', 'reservar ✅')
    return f'Reserva confirmada (ficticia): {vuelo} + {hotel}.'

print('Herramientas listas')

## Paso 2 · ✏️ Los tres agentes especialistas (huecos 1, 2 y 3)
**Copia de:** notebook 06, `crear_agente_creditos()` (paso 6), o notebook 06a, `crear_especialista()` (paso 8).

Cada agente recibe **solo sus herramientas** (contexto aislado). El Agente Reservas ya está hecho como ejemplo.

- **Hueco 1:** las herramientas del Agente Políticas (entre corchetes).
- **Hueco 2:** las instrucciones del Agente Clima (un texto entre comillas; dile qué es y que use su herramienta).
- **Hueco 3:** las herramientas del Agente Clima.

**📤 Salida esperada:** `Agentes listos` y una respuesta del Agente Políticas sobre cancelación (72 horas).

In [ ]:
def agente_reservas():
    return create_agent(model=nuevo_llm(), tools=[buscar_vuelos, buscar_hoteles],
        system_prompt='Eres el Agente de Reservas de una agencia de viajes ficticia. Busca vuelos y hoteles con tus herramientas. '
                      'Responde con nombres y precios exactos. Breve.')

def agente_politicas():
    return create_agent(model=nuevo_llm(),
        tools=[buscar_politica],                            # ✏️ COMPLETA (hueco 1)
        system_prompt='Eres el Agente de Políticas. Responde SOLO con lo que diga el manual (usa buscar_politica). '
                      'Si no está, di: "El manual no tiene esa información". Breve.')

def agente_clima():
    return create_agent(model=nuevo_llm(),
        tools=[consultar_clima],                            # ✏️ COMPLETA (hueco 3)
        system_prompt='Eres el Agente de Clima de una agencia de viajes ficticia. Usa consultar_clima para '
                      'responder sobre el clima del destino y el mes. Responde con el dato de la herramienta. Breve.')   # ✏️ COMPLETA (hueco 2)

print('Agentes listos')
print(agente_politicas().invoke({'messages': [('user', '¿Hasta cuándo puedo cancelar gratis?')]})['messages'][-1].text)

## Paso 3 · ✏️ Los subagentes como herramientas del Coordinador (huecos 4 y 5)
**Copia de:** notebook 06, `consultar_agente_creditos` (paso 9): allí el Asesor llamaba a otro agente a través de una herramienta. Aquí hacemos lo mismo, pero sin A2A.

`preguntar_a_reservas` ya está hecho. Escribe las otras dos **copiando y cambiando los nombres**:
- **Hueco 4:** dentro de `preguntar_a_politicas`, ¿qué función crea el agente?
- **Hueco 5:** dentro de `preguntar_a_clima`, ¿qué función crea el agente?

**📤 Salida esperada:** `Subagentes listos como herramientas: ['preguntar_a_reservas', 'preguntar_a_politicas', 'preguntar_a_clima']`

In [ ]:
@tool
def preguntar_a_reservas(pregunta: str) -> str:
    """Pregunta al Agente de Reservas por vuelos y hoteles con precios."""
    registrar('Coordinador', 'Agente Reservas', 'Delegación', pregunta[:35])
    r = agente_reservas().invoke({'messages': [('user', pregunta)]})['messages'][-1].text
    registrar('Agente Reservas', 'Coordinador', 'Resultado', 'responde')
    return r

@tool
def preguntar_a_politicas(pregunta: str) -> str:
    """Pregunta al Agente de Políticas por cancelación, equipaje o reembolsos."""
    registrar('Coordinador', 'Agente Políticas', 'Delegación', pregunta[:35])
    r = agente_politicas().invoke({'messages': [('user', pregunta)]})['messages'][-1].text   # ✏️ COMPLETA (hueco 4)
    registrar('Agente Políticas', 'Coordinador', 'Resultado', 'responde')
    return r

@tool
def preguntar_a_clima(pregunta: str) -> str:
    """Pregunta al Agente de Clima por el clima del destino."""
    registrar('Coordinador', 'Agente Clima', 'Delegación', pregunta[:35])
    r = agente_clima().invoke({'messages': [('user', pregunta)]})['messages'][-1].text       # ✏️ COMPLETA (hueco 5)
    registrar('Agente Clima', 'Coordinador', 'Resultado', 'responde')
    return r

print('Subagentes listos como herramientas:', [t.name for t in [preguntar_a_reservas, preguntar_a_politicas, preguntar_a_clima]])

## Paso 4 · ✏️ El Coordinador (hueco 6)
**Copia de:** notebook 06, `crear_asesor()` (paso 10): un agente que recibe una lista de herramientas.

- **Hueco 6:** la lista de herramientas del Coordinador: los **tres subagentes** y `calcular_presupuesto`.

**📤 Salida esperada:** una propuesta de viaje con vuelo, hotel, total, si cabe en los $3.000.000 y el clima; además el número de llamadas al LLM y tokens.

In [ ]:
PREGUNTA = ('Quiero ir a Cartagena 4 días (3 noches) en diciembre con mi pareja, presupuesto de 3.000.000. '
            '¿Qué me recomiendas? ¿Y si cancelo, qué pasa?')

def coordinador(pregunta):
    agente = create_agent(model=nuevo_llm(),
        tools=[preguntar_a_reservas, preguntar_a_politicas, preguntar_a_clima, calcular_presupuesto],   # ✏️ COMPLETA (hueco 6)
        system_prompt='Eres el Coordinador de ViajaFácil. Usa tus subagentes para vuelos, hoteles, políticas y clima. '
                      'Calcula el total SIEMPRE con calcular_presupuesto (nunca sumes tú). '
                      'Propón UNA opción de vuelo y hotel que quepa en el presupuesto. NO reserves: solo propón. Breve.')
    return agente.invoke({'messages': [('user', pregunta)]})['messages'][-1].text

BITACORA.clear(); CONTADOR.reiniciar()
registrar('Cliente', 'Coordinador', 'Usuario', 'pregunta')
propuesta = coordinador(PREGUNTA)
print(propuesta)
print(f'\n📊 {CONTADOR.llamadas} llamadas al LLM · {CONTADOR.tokens} tokens')

## Paso 5 · ✏️ Humano en el bucle: nadie reserva sin aprobación (hueco 7)
El Coordinador solo **propone**. Una persona decide. Solo si responde `si` se llama a `reservar()`.

- **Hueco 7:** la condición para reservar. Pista: compara la respuesta de la persona con `'si'`.

**📤 Salida esperada:** si escribes `si`, «Reserva confirmada (ficticia)…»; si escribes otra cosa, «Reserva cancelada por el cliente».

In [ ]:
print('PROPUESTA:\n', propuesta)
respuesta_humana = input('\n¿Aprueba la reserva? (si/no): ').strip().lower()

if respuesta_humana in ('si', 'sí'):   # ✏️ COMPLETA (hueco 7)
    registrar('Cliente', 'Coordinador', 'Usuario', '✋ aprueba')
    print(reservar('vuelo propuesto', 'hotel propuesto'))
else:
    registrar('Cliente', 'Coordinador', 'Usuario', '✋ rechaza')
    print('Reserva cancelada por el cliente. No se cobró nada.')

## Paso 6 · ▶️ Ver la bitácora en Gradio (ya está hecho)
Abre el enlace, escribe una pregunta y mira el diagrama: quién le habló a quién y cuántas llamadas costó.

In [ ]:
import gradio as gr

CARRILES = ['Cliente', 'Coordinador', 'Agente Reservas', 'Agente Políticas', 'Agente Clima']
COLORES = {'Usuario': '#2563eb', 'Delegación': '#7c3aed', 'Resultado': '#475569', 'Herramienta': '#0d9488'}

def dibujar(eventos):    # mismo estilo de diagrama de los notebooks 06 y 06a
    n = len(CARRILES)
    cab = ''.join(f'<div style="text-align:center;font-weight:700;color:#0f172a;padding:8px;background:#e2e8f0;border-radius:8px;font-size:13px">{c}</div>' for c in CARRILES)
    filas = ''
    for i, ev in enumerate(eventos, start=1):
        a, b = CARRILES.index(ev['de']), CARRILES.index(ev['para']); color = COLORES[ev['tipo']]
        if a == b:
            flecha = f'<div style="border:2px dashed {color};border-radius:8px;text-align:center;color:{color};font-weight:700;font-size:12px">↻ {ev["tipo"]}</div>'
        else:
            linea = f'<div style="flex:1;height:3px;background:{color}"></div>'
            flecha = f'<div style="display:flex;align-items:center;color:{color}">' + (linea + '▶' if b > a else '◀' + linea) + '</div>'
        filas += (f'<div style="grid-column:{min(a, b) + 1}/{max(a, b) + 2};padding:2px 8%"><div style="font-size:12px;color:{color};'
                  f'font-weight:600;text-align:center;white-space:nowrap;overflow:hidden;text-overflow:ellipsis">{i}. {ev["accion"]}</div>{flecha}</div>')
    return (f'<div style="background:#fff;border-radius:12px;padding:12px;color:#0f172a"><div style="display:grid;'
            f'grid-template-columns:repeat({n},minmax(0,1fr));gap:6px 0">{cab}{filas}</div></div>')

def atender(pregunta):
    BITACORA.clear(); CONTADOR.reiniciar()
    registrar('Cliente', 'Coordinador', 'Usuario', 'pregunta')
    try:
        texto = coordinador(pregunta)
    except Exception as error:
        texto = f'⛔ Error: {error}'
    registrar('Coordinador', 'Cliente', 'Usuario', 'propuesta')
    return dibujar(BITACORA), texto, f'### 📊 {CONTADOR.llamadas} llamadas al LLM · {CONTADOR.tokens} tokens'

with gr.Blocks(title='ViajaFácil') as demo:
    gr.Markdown('# ✈️ ViajaFácil\nCoordinador + 3 subagentes. Datos ficticios.')
    pregunta = gr.Textbox(label='Pregunta del cliente', value=PREGUNTA)
    gr.Examples([[PREGUNTA], ['¿Cuánto cuesta la maleta de bodega?'], ['¿Qué clima hace en Cartagena en diciembre?']], inputs=pregunta)
    boton = gr.Button('Preguntar', variant='primary')
    metricas = gr.Markdown(); diagrama = gr.HTML(); salida = gr.Textbox(label='Propuesta', lines=6)
    boton.click(atender, pregunta, [diagrama, salida, metricas])

demo.launch(share=True, debug=False)

## Paso 7 · ✏️ Para la socialización de las 8:00 p. m.
Prepara en 2 minutos:
1. Una captura de Gradio con la pregunta principal (diagrama + llamadas al LLM).
2. ¿Cuántas llamadas al LLM hizo tu sistema? ¿Cómo las reducirías? (Pista: ¿el Agente Clima necesita ser un agente?)
3. ¿Qué cambiarías para que `buscar_politica` fuera RAG de verdad, como en el 06a?

**Reto opcional:** convierte al Agente Clima en una **herramienta simple** (dale `consultar_clima` directo al Coordinador) y compara cuántas llamadas al LLM se ahorran.